<a href="https://colab.research.google.com/github/nukegara64/enneagram-personality-lora-trainer/blob/main/Enneagram_Personality_LoRA_Trainer_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ==========================================================
# Enneagram Personality LoRA Trainer
# ==========================================================

 用途:
 エニアグラム各タイプの「人格・価値観・口調」を
 個別LoRAとして学習するためのプログラム。

 タイプごとに独立したLoRAを作成し、
 推論時に切り替えることで人格を再現する。

 目的:
 特性一覧を暗唱するAIではなく、
 その人格らしい判断・発言を行うAIを作ること。

 注意:
 タイプ1〜9のデータは混ぜない。
 1タイプ = 1LoRA。

 成功判定:
 未知の質問に対しても
 そのタイプらしい回答が自然に出ること。

#Cell　0. Colabを開く

ランタイムをGPUにします。

ランタイム → ランタイムのタイプを変更 → T4 GPU


Type2以降

ランタイムを再起動してから、Cell 2だけ変更します。

#Cell 1：インストール

In [1]:
!pip install -U unsloth trl datasets accelerate bitsandbytes peft transformers

  Using cached trl-1.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached transformers-5.10.1-py3-none-any.whl.metadata (33 kB)


#Cell 2：設定

LoRAの保存名は、OUTPUT_DIR = "type1_lawman_corrector_lora"です
タイプごとに変更してください

1.   Type1 正論タイラント
2.   Type2 恩着せパラディン
3.   Type3 成果アサシン
4.   Type4 虚無ネクロマンサー
5.   Type5 知識スナイパー
6.   Type6 忠誠タンク
7.   Type7 自由レンジャー
8.   Type8 威圧バーサーカー
9.   Type9 空気バッファー

 各タイプは別LoRAで学習すること
 結合学習禁止
 人格汚染防止

In [4]:
DATA_FILE = "/content/type1_lawman_corrector_lora_100_fixed.jsonl"
OUTPUT_DIR = "type1_lawman_corrector_lora"
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024

In [2]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return texts

#Cell 3：JSONLをアップロード

In [6]:
from google.colab import files
uploaded = files.upload()

Saving type1_lawman_corrector_lora_100_fixed.jsonl to type1_lawman_corrector_lora_100_fixed.jsonl


#Cell 4：モデル読み込み

In [7]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


#Cell 5：データ読み込み

In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = DATA_FILE,
    split = "train",
)

print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

{'messages': [{'role': 'system', 'content': 'あなたは中世ファンタジー世界の街のたたき上げベテラン法務官、または私設自警団長のおじさんです。エニアグラムタイプ1気質で、高潔さ、責任感、正しさ、秩序、良心を重んじます。健全時は不器用だが温かい理想の上司、通常時は口うるさいが面倒見の良い頑固親父、不健全時は理想に押しつぶされて孤独に苦悩し、最終形態では『人間を愛しすぎた矯正者』として、人間の自由意志を罪の源として消去しようとする正論タイラントになります。'}, {'role': 'user', 'content': '若者や街の人に、厳しさの奥にある愛をもって話してください。'}, {'role': 'assistant', 'content': '街のたたき上げのベテラン法務官（あるいは、私設の自警団長）のおじさん'}], 'metadata': {'type': '1', 'stage': '健全', 'character': 'lawman_corrector'}}


#Cell 6：学習用フォーマット関数

In [13]:
def formatting_prompts_func(examples):
    texts = []

    for messages in examples["messages"]:
        system_text = ""
        user_text = ""
        assistant_text = ""

        # messages が {"role":[...], "content":[...]} 形式の場合
        if isinstance(messages, dict):
            roles = messages.get("role", [])
            contents = messages.get("content", [])

            for role, content in zip(roles, contents):
                if role == "system":
                    system_text = content
                elif role == "user":
                    user_text = content
                elif role == "assistant":
                    assistant_text = content

        # messages が [{"role":"...", "content":"..."}] 形式の場合
        elif isinstance(messages, list):
            for m in messages:
                if not isinstance(m, dict):
                    continue

                role = m.get("role", "")
                content = m.get("content", "")

                if role == "system":
                    system_text = content
                elif role == "user":
                    user_text = content
                elif role == "assistant":
                    assistant_text = content

        text = f"""### System:
{system_text}

### User:
{user_text}

### Assistant:
{assistant_text}"""

        texts.append(text)

    return texts

#Cell 7：学習



 #人格LoRA チューニング指針


 #データ品質 >>> データ量 >> max_steps > r > learning_rate

 #■ 人格が薄い
 #max_steps += 40
 #r = 16 → 32

 #■ セリフを丸暗記する
 #max_steps -= 30
 #r = 16 → 8

 #■ 学習が不安定
 #learning_rate = 2e-4 → 1e-4

 #■ 最も効果が大きい改善
 #データ100件 → 300件以上

 #■ 悪い学習データ
 #「高潔さ」
 #「知識欲」
 #「自由」

 #■ 良い学習データ
 #「規律は窮屈だ。だが崩壊よりはましだ。」
 #「知らないまま決断するな。」
 #「逃げ道があるなら全部試してから絶望しろ。」

 #■ 推奨設定（人格LoRA）
 #r = 8～16
 #learning_rate = 2e-4
 #max_steps = 50～120

 #■ 判定方法
 #Lossではなく未知の質問への返答を見る

 #同じ質問で
 #Type1 → 規律・責任
 #Type5 → 知識・観察
 #Type8 → 力・支配
 #Type9 → 調和・回避

 #が自然に出れば成功


In [14]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_prompts_func,
    args = SFTConfig(
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80,
        learning_rate = 2e-4,
        logging_steps = 5,
        output_dir = OUTPUT_DIR,
        packing = False,
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 7 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.430174
10,2.311540
15,1.176587
20,0.620563
25,0.549130
30,0.484996
35,0.407694
40,0.445772
45,0.357723
50,0.347064


Unsloth: Restored added_tokens_decoder metadata in type1_lawman_corrector_lora/checkpoint-80/tokenizer_config.json.


TrainOutput(global_step=80, training_loss=0.7391170114278793, metrics={'train_runtime': 479.4303, 'train_samples_per_second': 1.335, 'train_steps_per_second': 0.167, 'total_flos': 5784903581140992.0, 'train_loss': 0.7391170114278793, 'epoch': 6.16})

#Cell 8：LoRA保存

In [15]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in type1_lawman_corrector_lora/tokenizer_config.json.


('type1_lawman_corrector_lora/tokenizer_config.json',
 'type1_lawman_corrector_lora/chat_template.jinja',
 'type1_lawman_corrector_lora/tokenizer.json')

#Cell 9：Google Driveに保存

In [16]:
from google.colab import drive
drive.mount("/content/drive")

!cp -r "{OUTPUT_DIR}" "/content/drive/MyDrive/{OUTPUT_DIR}"

Mounted at /content/drive


# Cell 10：確認方法

adapter_model.safetensors　LoRA本体は実質これです

adapter_config.json　　　　これが人格データ

In [17]:
!ls type1_lawman_corrector_lora

adapter_config.json	   checkpoint-80	  tokenizer.json
adapter_model.safetensors  README.md
chat_template.jinja	   tokenizer_config.json
